# Revisions R2.3, R1.4, R1.1: final numbers for the paper (no GPU)

Run this **after** notebooks 01–05.

| Section | Answers |
|---|---|
| Mean ± SD over seeds + query-level bootstrap 95% CI per model | **R2.3** (*"confidence intervals calculated at the query level"*) |
| CI of the difference for each controlled comparison (decoder, scale, pretraining, NER checkpoint, supervision) | **R1.4** (effect of each component), **R2.2a/b/c** |
| Table 7 (M3 vs. GPT, span overlap) + cross-model false-negative overlap, recomputed from the same runs | own correction: the submitted Table 7 used a different M3 run |
| Run environment and time per model | **R1.1** (implementation table) |

**Runtime:** no GPU needed. Choose **CPU** (*Runtime → Change runtime type → CPU*) to save compute units.

In [ ]:
!pip install -q pytorch-crf seqeval openai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT = '/content/drive/MyDrive/strokeid_revision'   # <- same folder in every notebook
DATA, RUNS, RESULTS, CODE = f'{PROJECT}/data', f'{PROJECT}/runs', f'{PROJECT}/results', '/content/code'
for d in (RUNS, RESULTS, f'{CODE}/prompts'): os.makedirs(d, exist_ok=True)
sys.path.insert(0, CODE)
for f in ('train.tsv', 'val.tsv', 'test.tsv'):
    assert os.path.exists(f'{DATA}/{f}'), f'missing {DATA}/{f} (copies of the *_fixed.tsv files)'
print('setup OK')

In [ ]:
%%writefile /content/code/common.py
"""Shared helpers for the StrokeID-NER revision experiments.

Data format (one of):
  * CoNLL/TSV: one "token<TAB>label" per line, blank line between queries.
  * JSONL: one {"id": ..., "tokens": [...], "labels": [...]} object per line.
"""
import json
import os
from collections import Counter

ENTITY_TYPES = ["DGN", "RF", "SYM", "TMP"]
LABELS = ["O", "B-SYM", "I-SYM", "B-DGN", "I-DGN", "B-TMP", "I-TMP", "B-RF", "I-RF"]


def load_split(path):
    """Return a list of dicts: {id, tokens, labels}."""
    if path.endswith(".jsonl"):
        out = []
        with open(path, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if line.strip():
                    d = json.loads(line)
                    d.setdefault("id", i)
                    out.append(d)
        return out
    out, toks, labs = [], [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                if toks:
                    out.append({"id": len(out), "tokens": toks, "labels": labs})
                    toks, labs = [], []
                continue
            parts = line.split("\t") if "\t" in line else line.split()
            toks.append(parts[0])
            labs.append(parts[-1])
    if toks:
        out.append({"id": len(out), "tokens": toks, "labels": labs})
    return out


def find_split(data_dir, name):
    for ext in (".jsonl", ".tsv", ".conll", ".txt"):
        p = os.path.join(data_dir, name + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No {name}.(jsonl|tsv|conll|txt) in {data_dir}")


def get_spans(labels):
    """Extract (type, start, end_inclusive) spans from a BIO sequence.

    Mirrors seqeval's default (non-strict) behaviour: an I- tag that does not
    continue a span of the same type starts a new span.
    """
    spans, cur = [], None
    for i, lab in enumerate(list(labels) + ["O"]):
        tag, typ = (lab.split("-", 1) + [None])[:2] if lab != "O" else ("O", None)
        if cur is not None and (tag in ("O", "B") or typ != cur[0]):
            spans.append((cur[0], cur[1], i - 1))
            cur = None
        if tag == "B" or (tag == "I" and cur is None):
            cur = (typ, i)
    return spans


def per_query_counts(gold_seqs, pred_seqs, types=ENTITY_TYPES):
    """Array-like counts [n_queries][type] -> (tp, n_pred, n_gold), exact match."""
    rows = []
    for g, p in zip(gold_seqs, pred_seqs):
        gs, ps = set(get_spans(g)), set(get_spans(p))
        row = []
        for t in types:
            gt = {s for s in gs if s[0] == t}
            pt = {s for s in ps if s[0] == t}
            row.append((len(gt & pt), len(pt), len(gt)))
        rows.append(row)
    return rows


def macro_from_totals(tp, npred, ngold):
    """tp/npred/ngold are per-type sequences. Returns macro P, R, F1 and per-type F1
    (seqeval 'macro avg' definition: unweighted mean of per-type scores)."""
    ps, rs, fs = [], [], []
    for a, b, c in zip(tp, npred, ngold):
        p = a / b if b else 0.0
        r = a / c if c else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        ps.append(p); rs.append(r); fs.append(f)
    n = len(fs)
    return sum(ps) / n, sum(rs) / n, sum(fs) / n, fs


def write_predictions(path, examples, preds):
    with open(path, "w", encoding="utf-8") as f:
        for ex, p in zip(examples, preds):
            assert len(p) == len(ex["tokens"]), (ex["id"], len(p), len(ex["tokens"]))
            f.write(json.dumps({"id": ex["id"], "tokens": ex["tokens"],
                                "gold": ex["labels"], "pred": p}, ensure_ascii=False) + "\n")


def read_predictions(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def label_stats(examples):
    c = Counter()
    for ex in examples:
        for t, *_ in get_spans(ex["labels"]):
            c[t] += 1
    return dict(c)


In [ ]:
%%writefile /content/code/bootstrap_ci.py
"""Seed-averaged results + query-level bootstrap 95% CIs (R2.3).

For every model we
  1. report mean +/- SD of macro P/R/F1 and per-entity F1 over seeds;
  2. resample the 275 test QUERIES with replacement (B times); in each resample
     compute macro F1 for every seed and average over seeds -> percentile CI of
     the seed-averaged macro F1;
  3. for chosen model pairs, compute the paired difference on the same resamples
     -> CI of the difference, and the share of resamples in which the ranking flips.

Usage
  python bootstrap_ci.py --runs runs --models gpt0 gpt5 m1 m2 m4 m4crf m5 m5crf m3 m3crf \
      --pairs m3:m4 m3:m5 m5:m4 m2:m1 m4crf:m4 m3crf:m3 m5crf:m5 m1:gpt5 --B 10000 --out results
Each model directory must contain seed*/test_predictions.jsonl
(GPT runs: one 'seed' directory per prompt/run).
"""
import argparse
import glob
import json
import os

import numpy as np

from common import ENTITY_TYPES, macro_from_totals, per_query_counts, read_predictions


def load_model(run_dir):
    seeds = sorted(glob.glob(os.path.join(run_dir, "seed*", "test_predictions.jsonl")))
    if not seeds:
        raise FileNotFoundError(f"no seed*/test_predictions.jsonl under {run_dir}")
    mats, ids = [], None
    for p in seeds:
        rows = read_predictions(p)
        rows.sort(key=lambda r: str(r["id"]))
        cur = [str(r["id"]) for r in rows]
        if ids is None:
            ids = cur
        assert cur == ids, f"query ids differ across seeds in {run_dir}"
        mats.append(np.array(per_query_counts([r["gold"] for r in rows], [r["pred"] for r in rows])))
    return [os.path.basename(os.path.dirname(p)) for p in seeds], ids, np.stack(mats)  # [S, Q, T, 3]


def macro_f1_batch(counts, idx):
    """counts [S,Q,T,3], idx [B,Q] -> macro F1 [B,S]."""
    summed = counts[:, idx].sum(axis=2)                       # [S,B,T,3]
    tp, npred, ngold = summed[..., 0], summed[..., 1], summed[..., 2]
    p = np.divide(tp, npred, out=np.zeros_like(tp, dtype=float), where=npred > 0)
    r = np.divide(tp, ngold, out=np.zeros_like(tp, dtype=float), where=ngold > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    return f.mean(-1).T                                       # [B,S]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", required=True)
    ap.add_argument("--models", nargs="+", required=True)
    ap.add_argument("--pairs", nargs="*", default=[])
    ap.add_argument("--B", type=int, default=10000)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--out", default="results")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    data, ids0 = {}, None
    for m in a.models:
        seeds, ids, c = load_model(os.path.join(a.runs, m))
        if ids0 is None:
            ids0 = ids
        assert ids == ids0, f"{m}: test query ids differ from {a.models[0]}"
        data[m] = (seeds, c)
    Q = len(ids0)
    rng = np.random.default_rng(a.seed)
    idx = rng.integers(0, Q, size=(a.B, Q))

    summary, boot = {}, {}
    for m, (seeds, c) in data.items():
        per_seed = []
        for s in range(c.shape[0]):
            tot = c[s].sum(0)
            P, R, F, per = macro_from_totals(tot[:, 0], tot[:, 1], tot[:, 2])
            per_seed.append(dict(P=P, R=R, F1=F, **dict(zip(ENTITY_TYPES, per))))
        arr = {k: np.array([d[k] for d in per_seed]) for k in per_seed[0]}
        bs = []
        for start in range(0, a.B, 1000):           # chunk to bound memory
            bs.append(macro_f1_batch(c, idx[start:start + 1000]).mean(1))
        bs = np.concatenate(bs)
        boot[m] = bs
        summary[m] = dict(
            seeds=seeds, n_seeds=len(seeds),
            mean={k: float(v.mean()) for k, v in arr.items()},
            sd={k: float(v.std(ddof=1)) if len(v) > 1 else 0.0 for k, v in arr.items()},
            per_seed=per_seed,
            f1_ci95=[float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))])

    pairs = {}
    for pr in a.pairs:
        x, y = pr.split(":")
        d = boot[x] - boot[y]
        obs = summary[x]["mean"]["F1"] - summary[y]["mean"]["F1"]
        pairs[pr] = dict(diff=float(obs), ci95=[float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
                         share_resamples_reversed=float((d <= 0).mean() if obs > 0 else (d >= 0).mean()))

    json.dump(dict(n_queries=Q, B=a.B, models=summary, pairs=pairs),
              open(os.path.join(a.out, "bootstrap_summary.json"), "w"), indent=2)

    lines = ["| Model | Seeds | P | R | Macro F1 (mean ± SD) | 95% CI (query bootstrap) | " +
             " | ".join(ENTITY_TYPES) + " |", "|" + "---|" * (6 + len(ENTITY_TYPES))]
    for m, s in summary.items():
        mu, sd = s["mean"], s["sd"]
        lines.append(f"| {m} | {s['n_seeds']} | {mu['P']:.4f} | {mu['R']:.4f} | {mu['F1']:.4f} ± {sd['F1']:.4f} | "
                     f"[{s['f1_ci95'][0]:.4f}, {s['f1_ci95'][1]:.4f}] | " +
                     " | ".join(f"{mu[t]:.2f} ± {sd[t]:.2f}" for t in ENTITY_TYPES) + " |")
    if pairs:
        lines += ["", "| Comparison | ΔF1 | 95% CI of Δ | Share of resamples where ranking reverses |", "|---|---|---|---|"]
        for pr, d in pairs.items():
            lines.append(f"| {pr.replace(':', ' − ')} | {d['diff']:+.4f} | [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}] | "
                         f"{d['share_resamples_reversed']:.3f} |")
    open(os.path.join(a.out, "results_table.md"), "w").write("\n".join(lines) + "\n")
    print("\n".join(lines))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/code/span_overlap.py
"""Recompute Table 7 (span-level overlap, M3 vs GPT) and the cross-model
false-negative overlap from saved predictions, so both analyses use the SAME
runs as the main results table.

Usage
  # Table 7 for one seed / run pair
  python span_overlap.py --a runs/m3/seed42 --b runs/gpt0/seed42
  # every seed of A against every seed of B, then mean
  python span_overlap.py --a runs/m3 --b runs/gpt0 --all_seeds
  # false negatives missed by ALL listed models (seed 42 of each)
  python span_overlap.py --fn runs/m1/seed42 runs/m2/seed42 runs/m4/seed42 runs/m3/seed42
"""
import argparse
import glob
import os
from collections import Counter

from common import ENTITY_TYPES, get_spans, read_predictions


def load(run):
    p = run if run.endswith(".jsonl") else os.path.join(run, "test_predictions.jsonl")
    rows = sorted(read_predictions(p), key=lambda r: str(r["id"]))
    return rows


def overlap(a_rows, b_rows):
    c = Counter(); per = {t: Counter() for t in ENTITY_TYPES}
    for ra, rb in zip(a_rows, b_rows):
        assert ra["id"] == rb["id"], "different test queries"
        # gold label strings may differ by I-/B- at a span start (data versions); spans must agree
        assert set(get_spans(ra["gold"])) == set(get_spans(rb["gold"])), f"gold spans differ in query {ra['id']}"
        sa, sb = set(get_spans(ra["pred"])), set(get_spans(rb["pred"]))
        for s in get_spans(ra["gold"]):
            key = ("n1" if s in sa else "n0") + ("1" if s in sb else "0")
            c[key] += 1; per[s[0]][key] += 1
    return c, per


def show(c, per, name_a, name_b):
    n = sum(c.values())
    print(f"Gold spans: {n}")
    for k, lab in [("n11", "both correct"), ("n10", f"{name_a} only"), ("n01", f"{name_b} only"), ("n00", "both wrong")]:
        print(f"  {k} {lab:<22} {c[k]:5d}  {100*c[k]/n:5.1f}%")
    print(f"  ratio n10:n01 = {c['n10']/max(c['n01'],1):.1f}:1")
    for t in ENTITY_TYPES:
        p = per[t]
        print(f"    {t}: {name_a} only {p['n10']:4d} | {name_b} only {p['n01']:4d} | ratio {p['n10']/max(p['n01'],1):.1f}:1")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--a"); ap.add_argument("--b")
    ap.add_argument("--all_seeds", action="store_true")
    ap.add_argument("--fn", nargs="+", help="runs whose shared false negatives to count")
    o = ap.parse_args()
    if o.a and o.b:
        def nm(p):
            parts = p.rstrip("/").split("/")
            return "/".join(parts[-2:]) if parts[-1].startswith("seed") else parts[-1]
        na, nb = nm(o.a), nm(o.b)
        if o.all_seeds:
            tot = Counter(); k = 0
            for sa in sorted(glob.glob(os.path.join(o.a, "seed*"))):
                for sb in sorted(glob.glob(os.path.join(o.b, "seed*"))):
                    c, _ = overlap(load(sa), load(sb)); tot.update(c); k += 1
            print(f"Mean over {k} seed pairs:")
            for key in ("n11", "n10", "n01", "n00"):
                print(f"  {key} {tot[key]/k:.1f}")
        else:
            c, per = overlap(load(o.a), load(o.b)); show(c, per, na, nb)
    if o.fn:
        runs = [load(r) for r in o.fn]
        fn_any, fn_all = 0, 0
        typ_all = Counter(); own = [0] * len(runs)
        for rows in zip(*runs):
            gold = set(get_spans(rows[0]["gold"]))
            missed = [gold - set(get_spans(r["pred"])) for r in rows]
            union, inter = set().union(*missed), set.intersection(*missed)
            for k, m in enumerate(missed):
                own[k] += len(m)
            fn_any += len(union); fn_all += len(inter)
            for s in inter:
                typ_all[s[0]] += 1
        print(f"False-negative spans missed by at least one model: {fn_any}")
        print(f"Missed by ALL {len(runs)} models: {fn_all} ({100*fn_all/max(fn_any,1):.1f}% of the union)")
        print(f"  by type: {dict(typ_all)}")
        print("Same count as a share of EACH model's own false negatives:")
        for r, n in zip(o.fn, own):
            print(f"  {r}: {n} FNs -> {100*fn_all/max(n,1):.1f}% of them are missed by all {len(runs)} models")
        print("  NOTE: say in the paper which denominator is used.")


if __name__ == "__main__":
    main()


## R2.3 / R1.4: main table with confidence intervals and pairwise differences

In [ ]:
order = ['gpt0', 'gpt5', 'm1', 'm2', 'm4', 'm4crf', 'm5', 'm5crf', 'm3', 'm3crf']
import glob
have = [m for m in order if glob.glob(f'{RUNS}/{m}/seed*/test_predictions.jsonl')]
# Each pair = one controlled comparison (R1.4):
pairs = {'m2:m1': 'decoder (IndoBERT-base)', 'm4crf:m4': 'decoder (IndoBERT-large)', 'm5crf:m5': 'decoder (plain XLM-R)',
         'm3crf:m3': 'decoder (XLM-R NER ckpt)', 'm4:m1': 'scale (IndoBERT)', 'm5:m4': 'multilingual vs monolingual (large)',
         'm3:m5': 'Indonesian-NER checkpoint', 'm3:m4': 'submitted M3 vs M4', 'gpt5:gpt0': 'few-shot vs zero-shot',
         'm3:gpt5': 'fine-tuned best vs GPT 5-shot', 'm1:gpt5': 'smallest fine-tuned vs GPT 5-shot'}
use = [p for p in pairs if all(x in have for x in p.split(':'))]
print('models found:', have)
for p in use: print(f'  {p:<10} = {pairs[p]}')
!cd /content/code && python bootstrap_ci.py --runs "$RUNS" --models {' '.join(have)} --pairs {' '.join(use)} --B 10000 --out "$RESULTS"

## Own correction: Table 7 and the false-negative overlap, from the same runs

In [ ]:
!cd /content/code && python span_overlap.py --a "$RUNS/m3/seed42" --b "$RUNS/gpt0/seed42" | tee "$RESULTS/table7_seed42.txt"
!cd /content/code && python span_overlap.py --a "$RUNS/m3" --b "$RUNS/gpt0" --all_seeds | tee "$RESULTS/table7_all_seeds.txt"
!cd /content/code && python span_overlap.py --fn "$RUNS/m1/seed42" "$RUNS/m2/seed42" "$RUNS/m4/seed42" "$RUNS/m3/seed42" | tee "$RESULTS/fn_overlap_seed42.txt"

## R1.1: implementation details collected from the runs

In [ ]:
import json, glob, collections
rows = collections.defaultdict(list); env = set()
for p in glob.glob(f'{RUNS}/*/seed*/metrics.json'):
    m = json.load(open(p))
    if 'device' in m:
        rows[m['model']].append(m); env.add((m['device'], m['torch'], m['transformers']))
print('Environment (GPU, torch, transformers):', env)
print(f"{'model':<14}{'runs':>5}{'min/run':>9}{'best epochs':>14}{'truncated test queries':>24}")
for k in sorted(rows):
    r = rows[k]
    print(f"{k:<14}{len(r):>5}{sum(x['minutes'] for x in r)/len(r):>9.1f}{str([x['best_epoch'] for x in r]):>14}{str({x['truncated_queries']['test'] for x in r}):>24}")

In [ ]:
# Zip everything (predictions, metrics, result tables; no model weights) and send it to Claude
!cd "$PROJECT" && zip -qr revision_results.zip results runs -x "*/raw_responses.jsonl" && ls -lh revision_results.zip